## Load Relevant Packages 

In [ ]:
import numpy as np
import pandas as pd
import scipy
from scipy.stats import fisher_exact
import openpyxl
#import session_info
import os
import tqdm

In [ ]:
workspace_bucket = os.environ['WORKSPACE_BUCKET']

## Retrieve all eQTLs 

In [ ]:
#introns = [i for i in range(1, 27) if i != 16] # Create list of introns from 1 to 26, not including 16

#for x in introns: 
#    allele_table = pd.read_excel(f"Intron_{x}_Alleles_CF_Carriers.xlsx") # load alleles file
#    snps_table = pd.read_excel(f"{workspace_bucket}/data/Genotype Files/CF Carriers/Intron_{x}_CF_Carriers.xlsx") # load snp genotype call file
#    snps = allele_table['locus'] # load correct snp locus names
#    if snps.is_unique: # only proceeds if all snps are unique, personal coherency check 
#        new_columns = snps.tolist()+ snps_table.columns[-2:].tolist() # Adding F508del, V470M column
#        snps_table.columns = new_columns
#        snps_table.columns
#        snps_table.to_excel(f"{workspace_bucket}/data/Genotype Files/CF Carriers/adjusted/Intron_{x}_CF_Carriers.xlsx", index=False)
#    else:
#        print(f'error: values are not unique in intron {x}')

In [ ]:
introns = [i for i in range(1, 27) if i != 16] # Create list of introns from 1 to 26, not including 16

for y in introns:
    table = pd.read_excel(f"{workspace_bucket}/data/Genotype Files/CF Carriers/adjusted/Intron_{y}_CF_Carriers.xlsx")
    table['F508del & V470M Combination'] = 0
    
    for x in range(len(table)):
        if ((table.loc[x, 'V470M'] == 0) & (table.loc[x, 'F508del'] == 0)):
            table.loc[x, 'F508del & V470M Combination'] = 0
        elif ((table.loc[x, 'V470M'] == 1) & (table.loc[x, 'F508del'] == 0)):
            table.loc[x, 'F508del & V470M Combination'] = 1     
        elif((table.loc[x, 'V470M']== 2) & (table.loc[x, 'F508del'] == 0)):
            table.loc[x, 'F508del & V470M Combination'] = 2
        elif ((table.loc[x, 'V470M']== 2) & (table.loc[x, 'F508del'] == 1)):
            table.loc[x, 'F508del & V470M Combination'] = 3
        elif ((table.loc[x, 'V470M']== 2) & (table.loc[x, 'F508del'] == 2)):
            table.loc[x, 'F508del & V470M Combination'] = 4
        elif ((table.loc[x, 'V470M']== 1) & (table.loc[x, 'F508del'] == 1)):
            table.loc[x, 'F508del & V470M Combination'] = 5
    
    table.to_excel(f"{workspace_bucket}/data/Genotype Files/CF Carriers/adjusted/Intron_{y}_CF_Carriers.xlsx", index=False)

In [ ]:
introns = [i for i in range(1, 27) if i != 16] # Create list of introns from 1 to 26, not including 16

for y in introns:
    table = pd.read_excel(f"{workspace_bucket}/data/Genotype Files/CF Carriers/adjusted/Intron_{y}_CF_Carriers.xlsx")
    table['F508del & V470M Combination'] = 0
    
    for x in range(len(table)):
        if ((table.loc[x, 'V470M'] == 0) & (table.loc[x, 'F508del'] == 0)):
            table.loc[x, 'F508del & V470M Combination'] = 0
        elif ((table.loc[x, 'V470M'] == 1) & (table.loc[x, 'F508del'] == 0)):
            table.loc[x, 'F508del & V470M Combination'] = 1     
        elif((table.loc[x, 'V470M']== 2) & (table.loc[x, 'F508del'] == 0)):
            table.loc[x, 'F508del & V470M Combination'] = 2
        elif ((table.loc[x, 'V470M']== 2) & (table.loc[x, 'F508del'] == 1)):
            table.loc[x, 'F508del & V470M Combination'] = 3
        elif ((table.loc[x, 'V470M']== 2) & (table.loc[x, 'F508del'] == 2)):
            table.loc[x, 'F508del & V470M Combination'] = 4
        elif ((table.loc[x, 'V470M']== 1) & (table.loc[x, 'F508del'] == 1)):
            table.loc[x, 'F508del & V470M Combination'] = 5
        
    table.to_excel(f"{workspace_bucket}/data/Genotype Files/CF Carriers/adjusted/Intron_{y}_CF_Carriers.xlsx", index=False)

In [ ]:
# Upload all eQTLs found in CFTR Intronic Regions

#eQTL_file = pd.read_excel('All Intronic CFTR eQTLs | Master File.xlsx') Need to only run once
#eQTL_file = eQTL_file.to_excel(f'{workspace_bucket}/data/eQTL.xlsx') Need to only run onceb

eQTLs = pd.read_excel(f'{workspace_bucket}/data/eQTL.xlsx')
eQTL_catalogue = eQTLs[~(eQTLs['Ref'].isna())].reset_index(drop=True)
eQTLs = eQTLs[~(eQTLs['Ref'].isna())]['Ref']
len(eQTLs)


## F508del 

In [ ]:
# Check pwCF's F508del labelling
check_1 = pd.read_excel(f'{workspace_bucket}/data/Genotype Files/pwCF/adjusted/Intron_1.xlsx')
check_1['F508del'].value_counts()

In [ ]:
# Check CF Carrier's F508del labelling
check_1 = pd.read_excel(f'{workspace_bucket}/data/Genotype Files/CF Carriers/adjusted/Intron_1_CF_Carriers.xlsx')
check_1['F508del'].value_counts()

In [ ]:
introns = [i for i in range(1, 27) if i != 16] # Create list of introns from 1 to 26, not including 16

# Consolidate all predictions into one table 
for x in introns:
    if x == 1: 
        SNP_table_F508del = pd.read_excel(f'{workspace_bucket}/data/Genotype Files/pwCF/Files&Figures/Intron {x} | Final SNPs (0.7 Correlation Threshold) | F508del.xlsx') # Keeping all non-homozygous F508del relevant data
        SNP_table_F508del['intron'] = x 
    else: 
        temp_table = pd.read_excel(f'{workspace_bucket}/data/Genotype Files/pwCF/Files&Figures/Intron {x} | Final SNPs (0.7 Correlation Threshold) | F508del.xlsx') # Keeping all non-homozygous F508del relevant data
        temp_table['intron'] = x
        SNP_table_F508del = pd.concat([SNP_table_F508del,temp_table],axis=0)

In [ ]:
SNP_table_F508del['Significant? (No F508del)'].value_counts()
# Total statistically significant No F508del associations in pwCF are 134

In [ ]:
SNP_table_F508del['Significant? (F508del Heterozygous)'].value_counts()
# Total statistically significant Heterozygous F508del associations in pwCF are 105

In [ ]:
SNP_table_F508del['Significant? (F508del Homozygous)'].value_counts()
# Total statistically significant Homozygous F508del associations in pwCF are 11

In [ ]:
# Filter for SNPs which are either significant for No F508del Class, Heterozygous F508del Class, Homozygous F508del 
SNP_table_F508del = SNP_table_F508del[(SNP_table_F508del['Significant? (No F508del)'] == 'yes') | (SNP_table_F508del['Significant? (F508del Heterozygous)'] == 'yes') | (SNP_table_F508del['Significant? (F508del Homozygous)'] == 'yes')] # only keep SNPs with statistically significant associations 
SNP_table_F508del = SNP_table_F508del.reset_index(drop=True)
SNP_table_F508del

In [ ]:
# Perform Carrier Cohort Calculations
columns_to_keep = ()
Carrier_Comparison_F508del  = pd.DataFrame(columns= SNP_table_F508del.columns)
Carrier_Comparison_F508del

genotype_order_1 = ['SNP dosage associated with No F508del Mutation','SNP dosage associated with Heterozygous F508del Mutation']
genotype_order_2 = ['Odds-Ratio (No F508del)', 'Odds-Ratio (F508del Heterozygous)']
genotype_order_3 = ['P-Value (No F508del)', 'P-Value (F508del Heterozygous)']
genotype_order_4 = ['Significant? (No F508del)', 'Significant? (F508del Heterozygous)']

for x in range(len(SNP_table_F508del)):
    Carrier_Comparison_F508del.loc[x,'SNP Locus'] = SNP_table_F508del.loc[x, 'SNP Locus']
    Carrier_Comparison_F508del.loc[x,'Alleles'] = SNP_table_F508del.loc[x,'Alleles']
    Carrier_Comparison_F508del.loc[x,'Mean Contribution of Main SNP'] = SNP_table_F508del.loc[x,'Mean Contribution of Main SNP']
    Carrier_Comparison_F508del.loc[x,'Associated SNP Locus'] = SNP_table_F508del.loc[x,'Associated SNP Locus']
    Carrier_Comparison_F508del.loc[x,'Correlation'] = SNP_table_F508del.loc[x,'Correlation']
    Carrier_Comparison_F508del.loc[x,'SNP dosage associated with No F508del Mutation'] = SNP_table_F508del.loc[x,'SNP dosage associated with No F508del Mutation']
    Carrier_Comparison_F508del.loc[x,'SNP dosage associated with Heterozygous F508del Mutation'] = SNP_table_F508del.loc[x,'SNP dosage associated with Heterozygous F508del Mutation']
    Carrier_Comparison_F508del.loc[x,'intron'] = SNP_table_F508del.loc[x,'intron']

    for genotype in range(len(genotype_order_1)):

        if SNP_table_F508del.loc[x,genotype_order_4[genotype]] == 'yes':
            table = pd.read_excel(f"{workspace_bucket}/data/Genotype Files/CF Carriers/adjusted/Intron_{SNP_table_F508del.loc[x,'intron']}_CF_Carriers.xlsx")
            if SNP_table_F508del.loc[x,'SNP Locus']  not in table.columns:
                    Carrier_Comparison_F508del.loc[x,'SNP dosage associated with No F508del Mutation'] = 'Not found in Carrier Cohort'
                    Carrier_Comparison_F508del.loc[x,'SNP dosage associated with Heterozygous F508del Mutation'] = 'Not found in Carrier Cohort'
                    continue

            table = table[[SNP_table_F508del['SNP Locus'][x],'F508del']]
            combined_vals = table.value_counts() # get count of values 

            if SNP_table_F508del.loc[x,genotype_order_1[genotype]] == "No Polymorphism":
                a = combined_vals[(combined_vals.index.get_level_values(0) == 0) & (combined_vals.index.get_level_values(1) == genotype)].sum() # values where SNP is 0 and where there is F508del genotype of interest
                b = combined_vals[(combined_vals.index.get_level_values(0) == 0) & (combined_vals.index.get_level_values(1) != genotype)].sum() # values where SNP is 0 and no F508del genotype of interest
                c = combined_vals[(combined_vals.index.get_level_values(0) != 0) & (combined_vals.index.get_level_values(1) == genotype)].sum() # values where SNP is anything but 0 and there is F508del genotype of interest
                d = combined_vals[(combined_vals.index.get_level_values(0) != 0) & (combined_vals.index.get_level_values(1) != genotype)].sum() # values where SNP is anything but 0 and no F508del genotype of interest

            elif SNP_table_F508del.loc[x,genotype_order_1[genotype]] == "Heterozygous Polymorphism":
                a = combined_vals[(combined_vals.index.get_level_values(0) == 0.5) & (combined_vals.index.get_level_values(1) == genotype)].sum() # values where SNP is 0.5 and where there is F508del genotype of interest
                b = combined_vals[(combined_vals.index.get_level_values(0) == 0.5) & (combined_vals.index.get_level_values(1) != genotype)].sum() # values where SNP is 0.5 and no F508del genotype of interest
                c = combined_vals[(combined_vals.index.get_level_values(0) != 0.5) & (combined_vals.index.get_level_values(1) == genotype)].sum() # values where SNP is anything but 0.5 and there is F508del genotype of interest
                d = combined_vals[(combined_vals.index.get_level_values(0) != 0.5) & (combined_vals.index.get_level_values(1) != genotype)].sum() # values where SNP is anything but 0.5 and no F508del genotype of interest
         
            elif SNP_table_F508del.loc[x,genotype_order_1[genotype]] == "Homozygous Polymorphism":
                a = combined_vals[(combined_vals.index.get_level_values(0) == 1) & (combined_vals.index.get_level_values(1) == genotype)].sum() # values where SNP is 1 and where there is F508del genotype of interest
                b = combined_vals[(combined_vals.index.get_level_values(0) == 1) & (combined_vals.index.get_level_values(1) != genotype)].sum() # values where SNP is 1 and no F508del genotype of interest
                c = combined_vals[(combined_vals.index.get_level_values(0) != 1) & (combined_vals.index.get_level_values(1) == genotype)].sum() # values where SNP is anything but 1 and there is F508del genotype of interest
                d = combined_vals[(combined_vals.index.get_level_values(0) != 1) & (combined_vals.index.get_level_values(1) != genotype)].sum() # values where SNP is anything but 1 and no F508del genotype of interest

            table = np.array([[a,b], [c,d]])
            odds_ratio, p_value = fisher_exact(table,alternative='greater')
            if np.isinf(odds_ratio) or np.isnan(odds_ratio): # inf if b or c == 0 , nan if a or d = 0 while b or c = 0 
                if ((a == 0 and b == 0) or (c == 0 and d == 0) or (a == 0 and c == 0) or (b == 0 and d == 0)):
                    odds_ratio = 'Undefined (Unknown Direction)'
                elif (b == 0 or c == 0):
                    odds_ratio = '∞ (Strong Positive Association)'
            else:
                odds_ratio = float(f"{odds_ratio:.2f}")

            Carrier_Comparison_F508del.loc[x,genotype_order_2[genotype]] = odds_ratio
            Carrier_Comparison_F508del.loc[x,genotype_order_3[genotype]] = ("<1e-300" if p_value < 1e-300 
            else f"{p_value:.2e}" if p_value < 0.001 
            else f"{p_value:.3f}") ## Very small values due to high amounts of samples 

            if isinstance(odds_ratio, str):
                if ((odds_ratio == '∞ (Strong Positive Association)') and (float(p_value)< 0.05)):
                    Carrier_Comparison_F508del.loc[x,genotype_order_4[genotype]] = 'yes'
                else: 
                    Carrier_Comparison_F508del.loc[x,genotype_order_4[genotype]] = 'no'
            else: 
                if ((odds_ratio > 1) and (float(p_value) < 0.05)):
                    Carrier_Comparison_F508del.loc[x,genotype_order_4[genotype]] = 'yes'
                else: 
                    Carrier_Comparison_F508del.loc[x,genotype_order_4[genotype]] = 'no'

Carrier_Comparison_F508del

In [ ]:
# Create a combined table comparing results between pwCF and CF Carriers 
combined_table_f508del = pd.DataFrame(columns= ('SNP Locus', 'Alleles', 'Intron', 'Mean Contribution of Main SNP', 'Associated SNP Locus', 'Correlation', 'eQTL? (GTEx v9)', 'SNP ID (GTEx v9)', 'Gene to which the eQTL is associated', 'Tissue where the eQTL was identified', 'GTEx (v9) P-Value of eQTL', 'GTEx (v9) NES of eQTL', 'SNP found in carriers?', 
                                        'SNP dosage associated with No F508del Mutation', 'pwCF: Odds-Ratio (No F508del)', 'pwCF: P-Value (No F508del)', 'pwCF: Significant? (No F508del)', 'Carriers: Odds-Ratio (No F508del)', 'Carriers: P-Value (No F508del)', 'Carriers: Significant? (No F508del)',
                                        'SNP dosage associated with Heterozygous F508del Mutation', 'pwCF: Odds-Ratio (Heterozygous F508del)', 'pwCF: P-Value (Heterozygous F508del)', 'pwCF: Significant? (Heterozygous F508del)', 'Carriers: Odds-Ratio (Heterozygous F508del)', 'Carriers: P-Value (Heterozygous F508del)',
                                          'Carriers: Significant? (Heterozygous F508del)', 'SNP dosage associated with Homozygous F508del Mutation', 'pwCF: Odds-Ratio (Homozygous F508del)', 'pwCF: P-Value (Homozygous F508del)', 'pwCF: Significant? (Homozygous F508del)', 
                                          'Carriers: Odds-Ratio (Homozygous F508del)', 'Carriers: P-Value (Homozygous F508del)', 'Carriers: Significant? (Homozygous F508del)'))

combined_table_f508del['SNP Locus'] = SNP_table_F508del['SNP Locus']
combined_table_f508del['Alleles'] = SNP_table_F508del['Alleles']
combined_table_f508del['Intron'] = SNP_table_F508del['intron']
combined_table_f508del['Mean Contribution of Main SNP'] = SNP_table_F508del['Mean Contribution of Main SNP']
combined_table_f508del['Associated SNP Locus'] = SNP_table_F508del['Associated SNP Locus']
combined_table_f508del['Correlation'] = SNP_table_F508del['Correlation']
combined_table_f508del['SNP found in carriers?'] = Carrier_Comparison_F508del['SNP dosage associated with No F508del Mutation'].apply(lambda x: 'yes' if x != 'Not found in Carrier Cohort' else 'no')

combined_table_f508del['SNP dosage associated with No F508del Mutation'] = SNP_table_F508del['SNP dosage associated with No F508del Mutation']

combined_table_f508del['pwCF: Odds-Ratio (No F508del)'] = SNP_table_F508del['Odds-Ratio (No F508del)']
combined_table_f508del['pwCF: P-Value (No F508del)'] = SNP_table_F508del['P-Value (No F508del)']
combined_table_f508del['pwCF: Significant? (No F508del)'] = SNP_table_F508del['Significant? (No F508del)']

combined_table_f508del['Carriers: Odds-Ratio (No F508del)'] = Carrier_Comparison_F508del['Odds-Ratio (No F508del)']
combined_table_f508del['Carriers: P-Value (No F508del)'] = Carrier_Comparison_F508del['P-Value (No F508del)']
combined_table_f508del['Carriers: Significant? (No F508del)'] = Carrier_Comparison_F508del['Significant? (No F508del)']

combined_table_f508del['SNP dosage associated with Heterozygous F508del Mutation'] = SNP_table_F508del['SNP dosage associated with Heterozygous F508del Mutation']

combined_table_f508del['pwCF: Odds-Ratio (Heterozygous F508del)'] = SNP_table_F508del['Odds-Ratio (F508del Heterozygous)']
combined_table_f508del['pwCF: P-Value (Heterozygous F508del)'] = SNP_table_F508del['P-Value (F508del Heterozygous)']
combined_table_f508del['pwCF: Significant? (Heterozygous F508del)'] = SNP_table_F508del['Significant? (F508del Heterozygous)']

combined_table_f508del['Carriers: Odds-Ratio (Heterozygous F508del)'] = Carrier_Comparison_F508del['Odds-Ratio (F508del Heterozygous)']
combined_table_f508del['Carriers: P-Value (Heterozygous F508del)'] = Carrier_Comparison_F508del['P-Value (F508del Heterozygous)']
combined_table_f508del['Carriers: Significant? (Heterozygous F508del)'] = Carrier_Comparison_F508del['Significant? (F508del Heterozygous)']

combined_table_f508del['SNP dosage associated with Homozygous F508del Mutation'] = SNP_table_F508del['SNP dosage associated with Homozygous F508del Mutation']

combined_table_f508del['pwCF: Odds-Ratio (Homozygous F508del)'] = SNP_table_F508del['Odds-Ratio (F508del Homozygous)']
combined_table_f508del['pwCF: P-Value (Homozygous F508del)'] = SNP_table_F508del['P-Value (F508del Homozygous)']
combined_table_f508del['pwCF: Significant? (Homozygous F508del)'] = SNP_table_F508del['Significant? (F508del Homozygous)']

#combined_table_f508del['SNP Locus'] = combined_table_f508del['SNP Locus'].apply(lambda x: x.split(" ")[0])

In [ ]:
# Perform surrogate analysis for F508del Homozygous predictions by assessing association of the heterozygous genotype of the same SNP with heterozygous F508del status 

for x in range(len(combined_table_f508del)):
    if (combined_table_f508del.loc[x, 'pwCF: Significant? (Homozygous F508del)'] == 'yes') & (combined_table_f508del.loc[x, 'SNP dosage associated with Homozygous F508del Mutation'] == 'Homozygous Polymorphism'):
        table = pd.read_excel(f"{workspace_bucket}/data/Genotype Files/CF Carriers/adjusted/Intron_{combined_table_f508del.loc[x,'Intron']}_CF_Carriers.xlsx")
        if combined_table_f508del.loc[x,'SNP Locus']  not in table.columns:
                    combined_table_f508del.loc[x,'SNP dosage associated with No F508del Mutation'] = 'Not found in Carrier Cohort'
                    combined_table_f508del.loc[x,'SNP dosage associated with Heterozygous F508del Mutation'] = 'Not found in Carrier Cohort'
                    continue

        table = table[[SNP_table_F508del['SNP Locus'][x],'F508del']]
        combined_vals = table.value_counts() # get count of values 

        a = combined_vals[(combined_vals.index.get_level_values(0) == 0.5) & (combined_vals.index.get_level_values(1) == 1)].sum() # values where SNP is 0.5 and where there is F508del genotype of interest
        b = combined_vals[(combined_vals.index.get_level_values(0) == 0.5) & (combined_vals.index.get_level_values(1) != 1)].sum() # values where SNP is 0.5 and no F508del genotype of interest
        c = combined_vals[(combined_vals.index.get_level_values(0) != 0.5) & (combined_vals.index.get_level_values(1) == 1)].sum() # values where SNP is anything but 0.5 and there is F508del genotype of interest
        d = combined_vals[(combined_vals.index.get_level_values(0) != 0.5) & (combined_vals.index.get_level_values(1) != 1)].sum() # values where SNP is anything but 0.5 and no F508del genotype of interest

        table = np.array([[a,b], [c,d]])
        odds_ratio, p_value = fisher_exact(table,alternative='greater')
        if np.isinf(odds_ratio) or np.isnan(odds_ratio): # inf if b or c == 0 , nan if a or d = 0 while b or c = 0 
            if ((a == 0 and b == 0) or (c == 0 and d == 0) or (a == 0 and c == 0) or (b == 0 and d == 0)):
                odds_ratio = 'Undefined (Unknown Direction)'
            elif (b == 0 or c == 0):
                odds_ratio = '∞ (Strong Positive Association)'
        else:
            odds_ratio = float(f"{odds_ratio:.2f}")

        combined_table_f508del.loc[x,'Carriers: Odds-Ratio (Homozygous F508del)'] = odds_ratio
        combined_table_f508del.loc[x,'Carriers: P-Value (Homozygous F508del)'] = ("<1e-300" if p_value < 1e-300 
        else f"{p_value:.2e}" if p_value < 0.001 
        else f"{p_value:.3f}") ## Very small values due to high amounts of samples 

        if isinstance(odds_ratio, str):
            if ((odds_ratio == '∞ (Strong Positive Association)') and (float(p_value)< 0.05)):
                combined_table_f508del.loc[x,'Carriers: Significant? (Homozygous F508del)'] = 'yes'
            else: 
                combined_table_f508del.loc[x,'Carriers: Significant? (Homozygous F508del)'] = 'no'
        else: 
            if ((odds_ratio > 1) and (float(p_value) < 0.05)):
                combined_table_f508del.loc[x,'Carriers: Significant? (Homozygous F508del)'] = 'yes'
            else: 
                combined_table_f508del.loc[x,'Carriers: Significant? (Homozygous F508del)'] = 'no'

In [ ]:
# Check how many SNPs are missing in the Carrier Cohort (F508del)
combined_table_f508del['SNP found in carriers?'].value_counts()

In [ ]:
combined_table_f508del[combined_table_f508del['SNP found in carriers?'] == 'no']
# These are indeed not found in the carrier cohort alleles 

In [ ]:
# Check which SNPs were significant between pwCF and Carriers (No F508del Class)
combined_table_f508del[['pwCF: Significant? (No F508del)', 'Carriers: Significant? (No F508del)']].value_counts()

# Total SNPs validated in Carriers for association with No F508del are 102

In [ ]:
# Check which SNPs were significant between pwCF and Carriers (F508del Heterozygous Class)
combined_table_f508del[['pwCF: Significant? (Heterozygous F508del)', 'Carriers: Significant? (Heterozygous F508del)']].value_counts()

# Total SNPs validated in Carriers for association with Heterozygous F508del are 104

In [ ]:
# Check which SNPs were significant between pwCF and Carriers (F508del Homozygous Class)
combined_table_f508del[['pwCF: Significant? (Homozygous F508del)', 'Carriers: Significant? (Homozygous F508del)']].value_counts()

# Total SNPs validated in Carriers for association with Heterozygous F508del are 68

In [ ]:
# of F508del eQTLs
#condition_1 = (combined_table_f508del['pwCF: Significant? (No F508del)'] == 'yes') & (combined_table_f508del['Carriers: Significant? (No F508del)'] == 'yes')
condition_2 = (combined_table_f508del['pwCF: Significant? (Heterozygous F508del)'] == 'yes') & (combined_table_f508del['Carriers: Significant? (Heterozygous F508del)'] == 'yes')
condition_3 = (combined_table_f508del['pwCF: Significant? (Homozygous F508del)'] == 'yes') & (combined_table_f508del['Carriers: Significant? (Homozygous F508del)'] == 'yes')
F508del_eQTLs = combined_table_f508del[condition_2 | condition_3]
F508del_eQTLs = F508del_eQTLs['SNP Locus']
F508del_eQTLs = F508del_eQTLs[F508del_eQTLs.isin(eQTLs)]

# The F508del model finds associations with 99 intronic CFTR eQTLs
print(len(F508del_eQTLs))

In [ ]:
# Annotate combined table for eQTLs status 
combined_table_f508del['eQTL? (GTEx v9)'] = combined_table_f508del['SNP Locus'].isin(eQTLs).apply(lambda x: 'yes' if x == True else 'no')

# Coherency check: Making sure eQTL Alleles are correct
print(combined_table_f508del['eQTL? (GTEx v9)'].value_counts())

for x in range(len(combined_table_f508del)):
    if combined_table_f508del.loc[x, 'eQTL? (GTEx v9)'] == 'yes':
        snp_alleles = combined_table_f508del.loc[x, 'Alleles']
        ref_alleles = str(eQTL_catalogue[eQTL_catalogue['Ref'] == combined_table_f508del.loc[x,'SNP Locus']]['Alleles'].iloc[0])
        if snp_alleles != ref_alleles:
            combined_table_f508del.loc[x,'eQTL? (GTEx v9)'] = 'no'
        elif snp_alleles == ref_alleles:
            combined_table_f508del.loc[x,'SNP ID (GTEx v9)'] = str(eQTL_catalogue[eQTL_catalogue['Ref'] == combined_table_f508del.loc[x,'SNP Locus']]['SNP Id'].iloc[0])
            combined_table_f508del.loc[x,'Gene to which the eQTL is associated'] = str(eQTL_catalogue[eQTL_catalogue['Ref'] == combined_table_f508del.loc[x,'SNP Locus']]['Gene Symbol'].iloc[0])
            combined_table_f508del.loc[x,'Tissue where the eQTL was identified'] = str(eQTL_catalogue[eQTL_catalogue['Ref'] == combined_table_f508del.loc[x,'SNP Locus']]['Tissue'].iloc[0])
            combined_table_f508del.loc[x,'GTEx (v9) P-Value of eQTL'] = str(eQTL_catalogue[eQTL_catalogue['Ref'] == combined_table_f508del.loc[x,'SNP Locus']]['P-Value'].iloc[0])
            combined_table_f508del.loc[x,'GTEx (v9) NES of eQTL'] = str(eQTL_catalogue[eQTL_catalogue['Ref'] == combined_table_f508del.loc[x,'SNP Locus']]['NES'].iloc[0])

# Based on the printed values, the eQTL alleles are coherent 
print(combined_table_f508del['eQTL? (GTEx v9)'].value_counts())

In [ ]:
#combined_table_f508del.to_excel('F508del Associations | Master File.xlsx', index=False)
combined_table_f508del

## V470M

In [ ]:
# Check pwCF's V470M labelling
check_1 = pd.read_excel(f'{workspace_bucket}/data/Genotype Files/pwCF/adjusted/Intron_1.xlsx')
check_1['V470M'].value_counts()

In [ ]:
# Check CF Carrier's V470M labelling
check_1 = pd.read_excel(f'{workspace_bucket}/data/Genotype Files/CF Carriers/adjusted/Intron_1_CF_Carriers.xlsx')
check_1['V470M'].value_counts()

In [ ]:
introns = [i for i in range(1, 27) if i != 16] # Create list of introns from 1 to 26, not including 16

# Consolidate all predictions into one table 
for x in introns:
    if x == 1: 
        SNP_table_V470M = pd.read_excel(f'{workspace_bucket}/data/Genotype Files/pwCF/Files&Figures/Intron {x} | Final SNPs (0.7 Correlation Threshold) | V470M.xlsx')
        SNP_table_V470M['intron'] = x 
    else: 
        temp_table = pd.read_excel(f'{workspace_bucket}/data/Genotype Files/pwCF/Files&Figures/Intron {x} | Final SNPs (0.7 Correlation Threshold) | V470M.xlsx')
        temp_table['intron'] = x
        SNP_table_V470M = pd.concat([SNP_table_V470M,temp_table],axis=0)

In [ ]:
SNP_table_V470M['Significant? (V/V)'].value_counts()
# Total statistically significant V/V associations in pwCF are 123

In [ ]:
SNP_table_V470M['Significant? (V/M)'].value_counts()
# Total statistically significant V/M associations in pwCF are 64

In [ ]:
SNP_table_V470M['Significant? (M/M)'].value_counts()
# Total statistically significant V/V associations in pwCF are 90

In [ ]:
# Filter for SNPs which are either significant for any V470M Class
SNP_table_V470M = SNP_table_V470M[(SNP_table_V470M['Significant? (V/V)'] == 'yes') | (SNP_table_V470M['Significant? (V/M)'] == 'yes') | (SNP_table_V470M['Significant? (M/M)'] == 'yes')] # only keep SNPs with statistically significant associations 
SNP_table_V470M = SNP_table_V470M.reset_index(drop=True)
SNP_table_V470M

In [ ]:
# Perform Carrier Cohort Calculations

Carrier_Comparison_V470M  = pd.DataFrame(columns= SNP_table_V470M.columns)
Carrier_Comparison_V470M

genotype_order_1 = ['SNP dosage associated with V/V Polymorphism','SNP dosage associated with V/M Polymorphism', 'SNP dosage associated with M/M Polymorphism']
genotype_order_2 = ['Odds-Ratio (V/V)', 'Odds-Ratio (V/M)', 'Odds-Ratio (M/M)']
genotype_order_3 = ['P-Value (V/V)', 'P-Value (V/M)', 'P-Value (M/M)']
genotype_order_4 = ['Significant? (V/V)', 'Significant? (V/M)', 'Significant? (M/M)']

for x in range(len(SNP_table_V470M)):
    Carrier_Comparison_V470M.loc[x,'SNP Locus'] = SNP_table_V470M.loc[x, 'SNP Locus']
    Carrier_Comparison_V470M.loc[x,'Alleles'] = SNP_table_V470M.loc[x,'Alleles']
    Carrier_Comparison_V470M.loc[x,'Mean Contribution of Main SNP'] = SNP_table_V470M.loc[x,'Mean Contribution of Main SNP']
    Carrier_Comparison_V470M.loc[x,'Associated SNP Locus'] = SNP_table_V470M.loc[x,'Associated SNP Locus']
    Carrier_Comparison_V470M.loc[x,'Correlation'] = SNP_table_V470M.loc[x,'Correlation']
    Carrier_Comparison_V470M.loc[x,'SNP dosage associated with V/V Polymorphism'] = SNP_table_V470M.loc[x,'SNP dosage associated with V/V Polymorphism']
    Carrier_Comparison_V470M.loc[x,'SNP dosage associated with V/M Polymorphism'] = SNP_table_V470M.loc[x,'SNP dosage associated with V/M Polymorphism']
    Carrier_Comparison_V470M.loc[x,'SNP dosage associated with M/M Polymorphism'] = SNP_table_V470M.loc[x,'SNP dosage associated with M/M Polymorphism']
    Carrier_Comparison_V470M.loc[x,'intron'] = SNP_table_V470M.loc[x,'intron']

    for genotype in range(len(genotype_order_1)):

        if SNP_table_V470M.loc[x,genotype_order_4[genotype]] == 'yes':
            table = pd.read_excel(f"{workspace_bucket}/data/Genotype Files/CF Carriers/adjusted/Intron_{SNP_table_V470M.loc[x,'intron']}_CF_Carriers.xlsx")
            if SNP_table_V470M.loc[x,'SNP Locus']  not in table.columns:
                    Carrier_Comparison_V470M.loc[x,'SNP dosage associated with V/V Polymorphism'] = 'Not found in Carrier Cohort'
                    Carrier_Comparison_V470M.loc[x,'SNP dosage associated with V/M Polymorphism'] = 'Not found in Carrier Cohort'
                    Carrier_Comparison_V470M.loc[x,'SNP dosage associated with M/M Polymorphism'] = 'Not found in Carrier Cohort'
                    continue

            table = table[[SNP_table_V470M['SNP Locus'][x],'V470M']]
            combined_vals = table.value_counts() # get count of values 

            if SNP_table_V470M.loc[x,genotype_order_1[genotype]] == "No Polymorphism":
                a = combined_vals[(combined_vals.index.get_level_values(0) == 0) & (combined_vals.index.get_level_values(1) == genotype)].sum() # values where SNP is 0 and where there is V470M genotype of interest
                b = combined_vals[(combined_vals.index.get_level_values(0) == 0) & (combined_vals.index.get_level_values(1) != genotype)].sum() # values where SNP is 0 and no V470M genotype of interest
                c = combined_vals[(combined_vals.index.get_level_values(0) != 0) & (combined_vals.index.get_level_values(1) == genotype)].sum() # values where SNP is anything but 0 and there is V470M genotype of interest
                d = combined_vals[(combined_vals.index.get_level_values(0) != 0) & (combined_vals.index.get_level_values(1) != genotype)].sum() # values where SNP is anything but 0 and no V470M genotype of interest

            elif SNP_table_V470M.loc[x,genotype_order_1[genotype]] == "Heterozygous Polymorphism":
                a = combined_vals[(combined_vals.index.get_level_values(0) == 0.5) & (combined_vals.index.get_level_values(1) == genotype)].sum() # values where SNP is 0.5 and where there is V470M genotype of interest
                b = combined_vals[(combined_vals.index.get_level_values(0) == 0.5) & (combined_vals.index.get_level_values(1) != genotype)].sum() # values where SNP is 0.5 and no V470M genotype of interest
                c = combined_vals[(combined_vals.index.get_level_values(0) != 0.5) & (combined_vals.index.get_level_values(1) == genotype)].sum() # values where SNP is anything but 0.5 and there is V470M genotype of interest
                d = combined_vals[(combined_vals.index.get_level_values(0) != 0.5) & (combined_vals.index.get_level_values(1) != genotype)].sum() # values where SNP is anything but 0.5 and no V470M genotype of interest
         
            elif SNP_table_V470M.loc[x,genotype_order_1[genotype]] == "Homozygous Polymorphism":
                a = combined_vals[(combined_vals.index.get_level_values(0) == 1) & (combined_vals.index.get_level_values(1) == genotype)].sum() # values where SNP is 1 and where there is V470M genotype of interest
                b = combined_vals[(combined_vals.index.get_level_values(0) == 1) & (combined_vals.index.get_level_values(1) != genotype)].sum() # values where SNP is 1 and no V470M genotype of interest
                c = combined_vals[(combined_vals.index.get_level_values(0) != 1) & (combined_vals.index.get_level_values(1) == genotype)].sum() # values where SNP is anything but 1 and there is V470M genotype of interest
                d = combined_vals[(combined_vals.index.get_level_values(0) != 1) & (combined_vals.index.get_level_values(1) != genotype)].sum() # values where SNP is anything but 1 and no V470M genotype of interest

            table = np.array([[a,b], [c,d]])
            odds_ratio, p_value = fisher_exact(table,alternative='greater')
            if np.isinf(odds_ratio) or np.isnan(odds_ratio): # inf if b or c == 0 , nan if a or d = 0 while b or c = 0 
                if ((a == 0 and b == 0) or (c == 0 and d == 0) or (a == 0 and c == 0) or (b == 0 and d == 0)):
                    odds_ratio = 'Undefined (Unknown Direction)'
                elif (b == 0 or c == 0):
                    odds_ratio = '∞ (Strong Positive Association)'
            else:
                odds_ratio = float(f"{odds_ratio:.2f}")

            Carrier_Comparison_V470M.loc[x,genotype_order_2[genotype]] = odds_ratio
            Carrier_Comparison_V470M.loc[x,genotype_order_3[genotype]] = ("<1e-300" if p_value < 1e-300 
            else f"{p_value:.2e}" if p_value < 0.001 
            else f"{p_value:.3f}") ## Very small values due to high amounts of samples 

            if isinstance(odds_ratio, str):
                if ((odds_ratio == '∞ (Strong Positive Association)') and (float(p_value)< 0.05)):
                    Carrier_Comparison_V470M.loc[x,genotype_order_4[genotype]] = 'yes'
                else: 
                    Carrier_Comparison_V470M.loc[x,genotype_order_4[genotype]] = 'no'
            else: 
                if ((odds_ratio > 1) and (float(p_value) < 0.05)):
                    Carrier_Comparison_V470M.loc[x,genotype_order_4[genotype]] = 'yes'
                else: 
                    Carrier_Comparison_V470M.loc[x,genotype_order_4[genotype]] = 'no'

Carrier_Comparison_V470M

In [ ]:
# Create a combined table comparing results between pwCF and CF Carriers 
combined_table_V470M = pd.DataFrame(columns= ('SNP Locus', 'Alleles', 'Intron', 'Mean Contribution of Main SNP', 'Associated SNP Locus', 'Correlation', 'eQTL? (GTEx v9)', 'SNP ID (GTEx v9)', 'Gene to which the eQTL is associated', 'Tissue where the eQTL was identified', 'GTEx (v9) P-Value of eQTL', 'GTEx (v9) NES of eQTL', 'SNP found in carriers?', 
                                        'SNP dosage associated with V/V Polymorphism', 'pwCF: Odds-Ratio (V/V)', 'pwCF: P-Value (V/V)', 'pwCF: Significant? (V/V)', 'Carriers: Odds-Ratio (V/V)', 'Carriers: P-Value (V/V)', 'Carriers: Significant? (V/V)',
                                        'SNP dosage associated with V/M Polymorphism', 'pwCF: Odds-Ratio (V/M)', 'pwCF: P-Value (V/M)', 'pwCF: Significant? (V/M)', 'Carriers: Odds-Ratio (V/M)', 'Carriers: P-Value (V/M)', 'Carriers: Significant? (V/M)',
                                        'SNP dosage associated with M/M Polymorphism', 'pwCF: Odds-Ratio (M/M)', 'pwCF: P-Value (M/M)', 'pwCF: Significant? (M/M)', 'Carriers: Odds-Ratio (M/M)', 'Carriers: P-Value (M/M)', 'Carriers: Significant? (M/M)'))

combined_table_V470M['SNP Locus'] = SNP_table_V470M['SNP Locus']
combined_table_V470M['Alleles'] = SNP_table_V470M['Alleles']
combined_table_V470M['Intron'] = SNP_table_V470M['intron']
combined_table_V470M['Mean Contribution of Main SNP'] = SNP_table_V470M['Mean Contribution of Main SNP']
combined_table_V470M['Associated SNP Locus'] = SNP_table_V470M['Associated SNP Locus']
combined_table_V470M['Correlation'] = SNP_table_V470M['Correlation']
combined_table_V470M['SNP found in carriers?'] = Carrier_Comparison_V470M['SNP dosage associated with V/V Polymorphism'].apply(lambda x: 'yes' if x != 'Not found in Carrier Cohort' else 'no')

combined_table_V470M['SNP dosage associated with V/V Polymorphism'] = SNP_table_V470M['SNP dosage associated with V/V Polymorphism']

combined_table_V470M['pwCF: Odds-Ratio (V/V)'] = SNP_table_V470M['Odds-Ratio (V/V)']
combined_table_V470M['pwCF: P-Value (V/V)'] = SNP_table_V470M['P-Value (V/V)']
combined_table_V470M['pwCF: Significant? (V/V)'] = SNP_table_V470M['Significant? (V/V)']

combined_table_V470M['Carriers: Odds-Ratio (V/V)'] = Carrier_Comparison_V470M['Odds-Ratio (V/V)']
combined_table_V470M['Carriers: P-Value (V/V)'] = Carrier_Comparison_V470M['P-Value (V/V)']
combined_table_V470M['Carriers: Significant? (V/V)'] = Carrier_Comparison_V470M['Significant? (V/V)']

combined_table_V470M['SNP dosage associated with V/M Polymorphism'] = SNP_table_V470M['SNP dosage associated with V/M Polymorphism']

combined_table_V470M['pwCF: Odds-Ratio (V/M)'] = SNP_table_V470M['Odds-Ratio (V/M)']
combined_table_V470M['pwCF: P-Value (V/M)'] = SNP_table_V470M['P-Value (V/M)']
combined_table_V470M['pwCF: Significant? (V/M)'] = SNP_table_V470M['Significant? (V/M)']

combined_table_V470M['Carriers: Odds-Ratio (V/M)'] = Carrier_Comparison_V470M['Odds-Ratio (V/M)']
combined_table_V470M['Carriers: P-Value (V/M)'] = Carrier_Comparison_V470M['P-Value (V/M)']
combined_table_V470M['Carriers: Significant? (V/M)'] = Carrier_Comparison_V470M['Significant? (V/M)']

combined_table_V470M['SNP dosage associated with M/M Polymorphism'] = SNP_table_V470M['SNP dosage associated with M/M Polymorphism']

combined_table_V470M['pwCF: Odds-Ratio (M/M)'] = SNP_table_V470M['Odds-Ratio (M/M)']
combined_table_V470M['pwCF: P-Value (M/M)'] = SNP_table_V470M['P-Value (M/M)']
combined_table_V470M['pwCF: Significant? (M/M)'] = SNP_table_V470M['Significant? (M/M)']

combined_table_V470M['Carriers: Odds-Ratio (M/M)'] = Carrier_Comparison_V470M['Odds-Ratio (M/M)']
combined_table_V470M['Carriers: P-Value (M/M)'] = Carrier_Comparison_V470M['P-Value (M/M)']
combined_table_V470M['Carriers: Significant? (M/M)'] = Carrier_Comparison_V470M['Significant? (M/M)']

#combined_table_V470M['SNP Locus'] = combined_table_V470M['SNP Locus'].apply(lambda x: x.split(" ")[0])

In [ ]:
# Check how many SNPs are missing in the Carrier Cohort (V470M)
combined_table_V470M['SNP found in carriers?'].value_counts()

In [ ]:
combined_table_V470M[combined_table_V470M['SNP found in carriers?'] == 'no']
# These are indeed not found in the carrier cohort alleles 

In [ ]:
# Check which SNPs were significant between pwCF and Carriers (V/V Class)
combined_table_V470M[['pwCF: Significant? (V/V)', 'Carriers: Significant? (V/V)']].value_counts()

# Total SNPs validated in Carriers for association with V/V are 120

In [ ]:
# Check which SNPs were significant between pwCF and Carriers (V/M Class)
combined_table_V470M[['pwCF: Significant? (V/M)', 'Carriers: Significant? (V/M)']].value_counts()

# Total SNPs validated in Carriers for association with V/M are 62

In [ ]:
# Check which SNPs were significant between pwCF and Carriers (M/M Class)
combined_table_V470M[['pwCF: Significant? (M/M)', 'Carriers: Significant? (M/M)']].value_counts()

# Total SNPs validated in Carriers for association with M/M are 84

In [ ]:
# of V470M eQTLs
condition_1 = (combined_table_V470M['pwCF: Significant? (V/V)'] == 'yes') & (combined_table_V470M['Carriers: Significant? (V/V)'] == 'yes')
condition_2 = (combined_table_V470M['pwCF: Significant? (V/M)'] == 'yes') & (combined_table_V470M['Carriers: Significant? (V/M)'] == 'yes')
condition_3 = (combined_table_V470M['pwCF: Significant? (M/M)'] == 'yes') & (combined_table_V470M['Carriers: Significant? (M/M)'] == 'yes')
V470M_eQTLs = combined_table_V470M[condition_1 | condition_2 | condition_3]
V470M_eQTLs = V470M_eQTLs['SNP Locus']
V470M_eQTLs = V470M_eQTLs[V470M_eQTLs.isin(eQTLs)]

# The V470M model finds associations with 99 intronic CFTR eQTLs
print(len(V470M_eQTLs))

In [ ]:
## Add SNPs
combined_table_V470M['eQTL? (GTEx v9)'] = combined_table_V470M['SNP Locus'].isin(eQTLs).apply(lambda x: 'yes' if x == True else 'no')

# Coherency check: Making sure eQTL Alleles are correct and applying SNP names 
print(combined_table_V470M['eQTL? (GTEx v9)'].value_counts())

for x in range(len(combined_table_V470M)):
    if combined_table_V470M.loc[x, 'eQTL? (GTEx v9)'] == 'yes':
        snp_alleles = combined_table_V470M.loc[x, 'Alleles']
        ref_alleles = str(eQTL_catalogue[eQTL_catalogue['Ref'] == combined_table_V470M.loc[x,'SNP Locus']]['Alleles'].iloc[0])
        if snp_alleles != ref_alleles:
            combined_table_V470M.loc[x,'eQTL? (GTEx v9)'] = 'no'
        elif snp_alleles == ref_alleles:
            combined_table_V470M.loc[x,'SNP ID (GTEx v9)'] = str(eQTL_catalogue[eQTL_catalogue['Ref'] == combined_table_V470M.loc[x,'SNP Locus']]['SNP Id'].iloc[0])
            combined_table_V470M.loc[x,'Gene to which the eQTL is associated'] = str(eQTL_catalogue[eQTL_catalogue['Ref'] == combined_table_V470M.loc[x,'SNP Locus']]['Gene Symbol'].iloc[0])
            combined_table_V470M.loc[x,'Tissue where the eQTL was identified'] = str(eQTL_catalogue[eQTL_catalogue['Ref'] == combined_table_V470M.loc[x,'SNP Locus']]['Tissue'].iloc[0])
            combined_table_V470M.loc[x,'GTEx (v9) P-Value of eQTL'] = str(eQTL_catalogue[eQTL_catalogue['Ref'] == combined_table_V470M.loc[x,'SNP Locus']]['P-Value'].iloc[0])
            combined_table_V470M.loc[x,'GTEx (v9) NES of eQTL'] = str(eQTL_catalogue[eQTL_catalogue['Ref'] == combined_table_V470M.loc[x,'SNP Locus']]['NES'].iloc[0])

print(combined_table_V470M['eQTL? (GTEx v9)'].value_counts())

In [ ]:
#combined_table_V470M.to_excel('V470M Associations | Master File.xlsx', index=False)
combined_table_V470M

## V470M + F508del

In [ ]:
# check class distribution in pwCF cohort
check_0 = pd.read_excel(f'{workspace_bucket}/data/Genotype Files/pwCF/adjusted/Intron_1.xlsx')

# Assign a F508del & V470M synergistic classification to each sample (+ Remove any individual data that is missing target variable data)
check_0 = check_0[pd.notna(check_0['V470M'])]
check_0=check_0.reset_index(drop=True)
check_01 = check_0
check_01['F508del & V470M Combination'] = 0
check_01

for x in range(check_0.shape[0]):
    if ((check_0.loc[x, 'V470M'] == 0) & (check_0.loc[x, 'F508del'] == 0)):
        check_01.loc[x, 'F508del & V470M Combination'] = 0
    elif ((check_0.loc[x, 'V470M'] == 1) & (check_0.loc[x, 'F508del'] == 0)):
        check_01.loc[x, 'F508del & V470M Combination'] = 1
    elif((check_0.loc[x, 'V470M']== 2) & (check_0.loc[x, 'F508del'] == 0)):
        check_01.loc[x, 'F508del & V470M Combination'] = 2
    elif ((check_0.loc[x, 'V470M']== 2) & (check_0.loc[x, 'F508del'] == 1)):
        check_01.loc[x, 'F508del & V470M Combination'] = 3
    elif ((check_0.loc[x, 'V470M']== 2) & (check_0.loc[x, 'F508del'] == 2)):
        check_01.loc[x, 'F508del & V470M Combination'] = 4
    elif ((check_0.loc[x, 'V470M']== 1) & (check_0.loc[x, 'F508del'] == 1)):
        check_01.loc[x, 'F508del & V470M Combination'] = 5

check_0['F508del & V470M Combination'].value_counts()

In [ ]:
# check class distribution in pwCF cohort
check_0 = pd.read_excel(f'{workspace_bucket}/data/Genotype Files/CF Carriers/adjusted/Intron_1_CF_Carriers.xlsx')

# Assign a F508del & V470M synergistic classification to each sample (+ Remove any individual data that is missing target variable data)
check_0 = check_0[pd.notna(check_0['V470M'])]
check_0=check_0.reset_index(drop=True)
check_01 = check_0
check_01['F508del & V470M Combination'] = 0
check_01

for x in range(check_0.shape[0]):
    if ((check_0.loc[x, 'V470M'] == 0) & (check_0.loc[x, 'F508del'] == 0)):
        check_01.loc[x, 'F508del & V470M Combination'] = 0
    elif ((check_0.loc[x, 'V470M'] == 1) & (check_0.loc[x, 'F508del'] == 0)):
        check_01.loc[x, 'F508del & V470M Combination'] = 1
    elif((check_0.loc[x, 'V470M']== 2) & (check_0.loc[x, 'F508del'] == 0)):
        check_01.loc[x, 'F508del & V470M Combination'] = 2
    elif ((check_0.loc[x, 'V470M']== 2) & (check_0.loc[x, 'F508del'] == 1)):
        check_01.loc[x, 'F508del & V470M Combination'] = 3
    elif ((check_0.loc[x, 'V470M']== 2) & (check_0.loc[x, 'F508del'] == 2)):
        check_01.loc[x, 'F508del & V470M Combination'] = 4
    elif ((check_0.loc[x, 'V470M']== 1) & (check_0.loc[x, 'F508del'] == 1)):
        check_01.loc[x, 'F508del & V470M Combination'] = 5

check_0['F508del & V470M Combination'].value_counts()

In [ ]:
# Consolidate all predictions into one table 
for x in introns:
    if x == 1: 
        SNP_table_Combined_Model = pd.read_excel(f'{workspace_bucket}/data/Genotype Files/pwCF/Files&Figures/Intron {x} | Final SNPs (0.7 Correlation Threshold) | V470M+F508del.xlsx')
        SNP_table_Combined_Model['intron'] = x 
    else: 
        temp_table = pd.read_excel(f'{workspace_bucket}/data/Genotype Files/pwCF/Files&Figures/Intron {x} | Final SNPs (0.7 Correlation Threshold) | V470M+F508del.xlsx')
        temp_table['intron'] = x
        SNP_table_Combined_Model = pd.concat([SNP_table_Combined_Model,temp_table],axis=0)

In [ ]:
SNP_table_Combined_Model['Significant? (V/V & No F508del)'].value_counts()
# Total statistically significant V/V & No F508del associations are 78

In [ ]:
SNP_table_Combined_Model['Significant? (V/M & No F508del)'].value_counts()
# Total statistically significant V/M & No F508del associations are 100

In [ ]:
SNP_table_Combined_Model['Significant? (M/M & No F508del)'].value_counts()
# Total statistically significant M/M & No F508del associations are 101

In [ ]:
SNP_table_Combined_Model['Significant? (M/M & Heterozygous F508del)'].value_counts()
# Total statistically significant M/M & Heterozygous F508del associations are 140

In [ ]:
SNP_table_Combined_Model['Significant? (M/M & Homozygous F508del)'].value_counts()
# Total statistically significant M/M & Homozygous F508del associations are 96

In [ ]:
SNP_table_Combined_Model['Significant? (V/M & Heterozygous F508del)'].value_counts()
# Total statistically significant V/M & Heterozygous F508del associations are 114

In [ ]:
# Filter for SNPs which are either significant for any Combined Class
SNP_table_Combined_Model = SNP_table_Combined_Model[(SNP_table_Combined_Model['Significant? (V/V & No F508del)'] == 'yes') | (SNP_table_Combined_Model['Significant? (V/M & No F508del)'] == 'yes') | (SNP_table_Combined_Model['Significant? (M/M & No F508del)'] == 'yes') |
                                                    (SNP_table_Combined_Model['Significant? (M/M & Heterozygous F508del)'] == 'yes') | (SNP_table_Combined_Model['Significant? (M/M & Homozygous F508del)'] == 'yes') | (SNP_table_Combined_Model['Significant? (V/M & Heterozygous F508del)'] == 'yes') ] 
SNP_table_Combined_Model = SNP_table_Combined_Model.reset_index(drop=True)
SNP_table_Combined_Model

In [ ]:
# Perform Carrier Cohort Calculations

Carrier_Comparison_Combined_Model  = pd.DataFrame(columns= SNP_table_Combined_Model.columns)
Carrier_Comparison_Combined_Model

genotype_order_1 = ['SNP dosage associated with V/V & No F508del Mutation','SNP dosage associated with V/M & No F508del Mutation', 'SNP dosage associated with M/M & No F508del Mutation',
                    'SNP dosage associated with M/M & Heterozygous F508del Mutation','SNP dosage associated with M/M & Homozygous F508del Mutation', 'SNP dosage associated with V/M & Heterozygous F508del Mutation']

genotype_order_2 = ['Odds-Ratio (V/V & No F508del)', 'Odds-Ratio (V/M & No F508del)', 'Odds-Ratio (M/M & No F508del)',
                    'Odds-Ratio (M/M & Heterozygous F508del)', 'Odds-Ratio (M/M & Homozygous F508del)', 'Odds-Ratio (V/M & Heterozygous F508del)']

genotype_order_3 = ['P-Value (V/V & No F508del)', 'P-Value (V/M & No F508del)', 'P-Value (M/M & No F508del)',
                    'P-Value (M/M & Heterozygous F508del)', 'P-Value (M/M & Homozygous F508del)', 'P-Value (V/M & Heterozygous F508del)']

genotype_order_4 = ['Significant? (V/V & No F508del)', 'Significant? (V/M & No F508del)', 'Significant? (M/M & No F508del)',
                    'Significant? (M/M & Heterozygous F508del)', 'Significant? (M/M & Homozygous F508del)', 'Significant? (V/M & Heterozygous F508del)']

for x in range(len(SNP_table_Combined_Model)):
    Carrier_Comparison_Combined_Model.loc[x,'SNP Locus'] = SNP_table_Combined_Model.loc[x, 'SNP Locus']
    Carrier_Comparison_Combined_Model.loc[x,'Alleles'] = SNP_table_Combined_Model.loc[x,'Alleles']
    Carrier_Comparison_Combined_Model.loc[x,'Mean Contribution of Main SNP'] = SNP_table_Combined_Model.loc[x,'Mean Contribution of Main SNP']
    Carrier_Comparison_Combined_Model.loc[x,'Associated SNP Locus'] = SNP_table_Combined_Model.loc[x,'Associated SNP Locus']
    Carrier_Comparison_Combined_Model.loc[x,'Correlation'] = SNP_table_Combined_Model.loc[x,'Correlation']
    Carrier_Comparison_Combined_Model.loc[x,'SNP dosage associated with V/V & No F508del Mutation'] = SNP_table_Combined_Model.loc[x,'SNP dosage associated with V/V & No F508del Mutation']
    Carrier_Comparison_Combined_Model.loc[x,'SNP dosage associated with V/M & No F508del Mutation'] = SNP_table_Combined_Model.loc[x,'SNP dosage associated with V/M & No F508del Mutation']
    Carrier_Comparison_Combined_Model.loc[x,'SNP dosage associated with M/M & No F508del Mutation'] = SNP_table_Combined_Model.loc[x,'SNP dosage associated with M/M & No F508del Mutation']
    Carrier_Comparison_Combined_Model.loc[x,'SNP dosage associated with M/M & Heterozygous F508del Mutation'] = SNP_table_Combined_Model.loc[x,'SNP dosage associated with M/M & Heterozygous F508del Mutation']
    Carrier_Comparison_Combined_Model.loc[x,'SNP dosage associated with M/M & Homozygous F508del Mutation'] = SNP_table_Combined_Model.loc[x,'SNP dosage associated with M/M & Homozygous F508del Mutation']
    Carrier_Comparison_Combined_Model.loc[x,'SNP dosage associated with V/M & Heterozygous F508del Mutation'] = SNP_table_Combined_Model.loc[x,'SNP dosage associated with V/M & Heterozygous F508del Mutation']
    Carrier_Comparison_Combined_Model.loc[x,'intron'] = SNP_table_Combined_Model.loc[x,'intron']

    for genotype in range(len(genotype_order_1)):

        if SNP_table_Combined_Model.loc[x,genotype_order_4[genotype]] == 'yes':
            table = pd.read_excel(f"{workspace_bucket}/data/Genotype Files/CF Carriers/adjusted/Intron_{SNP_table_Combined_Model.loc[x,'intron']}_CF_Carriers.xlsx")
            if SNP_table_Combined_Model.loc[x,'SNP Locus']  not in table.columns:
                    Carrier_Comparison_Combined_Model.loc[x,'SNP dosage associated with V/V & No F508del Mutation'] = 'Not found in Carrier Cohort'
                    Carrier_Comparison_Combined_Model.loc[x,'SNP dosage associated with V/M & No F508del Mutation'] = 'Not found in Carrier Cohort'
                    Carrier_Comparison_Combined_Model.loc[x,'SNP dosage associated with M/M & No F508del Mutation'] = 'Not found in Carrier Cohort'
                    Carrier_Comparison_Combined_Model.loc[x,'SNP dosage associated with M/M & Heterozygous F508del Mutation'] = 'Not found in Carrier Cohort'
                    Carrier_Comparison_Combined_Model.loc[x,'SNP dosage associated with M/M & Homozygous F508del Mutation'] = 'Not found in Carrier Cohort'
                    Carrier_Comparison_Combined_Model.loc[x,'SNP dosage associated with V/M & Heterozygous F508del Mutation'] = 'Not found in Carrier Cohort'
                    continue

            table = table[[SNP_table_Combined_Model['SNP Locus'][x],'F508del & V470M Combination']]
            combined_vals = table.value_counts() # get count of values 

            if SNP_table_Combined_Model.loc[x,genotype_order_1[genotype]] == "No Polymorphism":
                if genotype == 4:
                    # skip testing M/M & Homozygous F508del Mutation due to no samples
                    continue
                a = combined_vals[(combined_vals.index.get_level_values(0) == 0) & (combined_vals.index.get_level_values(1) == genotype)].sum() # values where SNP is 0 and where there is V470M genotype of interest
                b = combined_vals[(combined_vals.index.get_level_values(0) == 0) & (combined_vals.index.get_level_values(1) != genotype)].sum() # values where SNP is 0 and no V470M genotype of interest
                c = combined_vals[(combined_vals.index.get_level_values(0) != 0) & (combined_vals.index.get_level_values(1) == genotype)].sum() # values where SNP is anything but 0 and there is V470M genotype of interest
                d = combined_vals[(combined_vals.index.get_level_values(0) != 0) & (combined_vals.index.get_level_values(1) != genotype)].sum() # values where SNP is anything but 0 and no V470M genotype of interest

            elif SNP_table_Combined_Model.loc[x,genotype_order_1[genotype]] == "Heterozygous Polymorphism":
                if genotype == 4:
                    # skip testing M/M & Homozygous F508del Mutation due to no samples
                    continue
                a = combined_vals[(combined_vals.index.get_level_values(0) == 0.5) & (combined_vals.index.get_level_values(1) == genotype)].sum() # values where SNP is 0.5 and where there is V470M genotype of interest
                b = combined_vals[(combined_vals.index.get_level_values(0) == 0.5) & (combined_vals.index.get_level_values(1) != genotype)].sum() # values where SNP is 0.5 and no V470M genotype of interest
                c = combined_vals[(combined_vals.index.get_level_values(0) != 0.5) & (combined_vals.index.get_level_values(1) == genotype)].sum() # values where SNP is anything but 0.5 and there is V470M genotype of interest
                d = combined_vals[(combined_vals.index.get_level_values(0) != 0.5) & (combined_vals.index.get_level_values(1) != genotype)].sum() # values where SNP is anything but 0.5 and no V470M genotype of interest
         
            elif SNP_table_Combined_Model.loc[x,genotype_order_1[genotype]] == "Homozygous Polymorphism":
                if genotype == 4: 
                    # testing M/M & Homozygous F508del Mutation as V/M & Heterozygous F508del Mutation (which acts as a surrogate) due to no samples 
                    a = combined_vals[(combined_vals.index.get_level_values(0) == 0.5) & (combined_vals.index.get_level_values(1) == 5)].sum() # values where SNP is 1 and where there is V470M genotype of interest
                    b = combined_vals[(combined_vals.index.get_level_values(0) == 0.5) & (combined_vals.index.get_level_values(1) != 5)].sum() # values where SNP is 1 and no V470M genotype of interest
                    c = combined_vals[(combined_vals.index.get_level_values(0) != 0.5) & (combined_vals.index.get_level_values(1) == 5)].sum() # values where SNP is anything but 1 and there is V470M genotype of interest
                    d = combined_vals[(combined_vals.index.get_level_values(0) != 0.5) & (combined_vals.index.get_level_values(1) != 5)].sum() # values where SNP is anything but 1 and no V470M genotype of interest
                else:  
                    a = combined_vals[(combined_vals.index.get_level_values(0) == 1) & (combined_vals.index.get_level_values(1) == genotype)].sum() # values where SNP is 1 and where there is V470M genotype of interest
                    b = combined_vals[(combined_vals.index.get_level_values(0) == 1) & (combined_vals.index.get_level_values(1) != genotype)].sum() # values where SNP is 1 and no V470M genotype of interest
                    c = combined_vals[(combined_vals.index.get_level_values(0) != 1) & (combined_vals.index.get_level_values(1) == genotype)].sum() # values where SNP is anything but 1 and there is V470M genotype of interest
                    d = combined_vals[(combined_vals.index.get_level_values(0) != 1) & (combined_vals.index.get_level_values(1) != genotype)].sum() # values where SNP is anything but 1 and no V470M genotype of interest

            table = np.array([[a,b], [c,d]])
            odds_ratio, p_value = fisher_exact(table,alternative='greater')
            if np.isinf(odds_ratio) or np.isnan(odds_ratio): # inf if b or c == 0 , nan if a or d = 0 while b or c = 0 
                if ((a == 0 and b == 0) or (c == 0 and d == 0) or (a == 0 and c == 0) or (b == 0 and d == 0)):
                    odds_ratio = 'Undefined (Unknown Direction)'
                elif (b == 0 or c == 0):
                    odds_ratio = '∞ (Strong Positive Association)'
            else:
                odds_ratio = float(f"{odds_ratio:.2f}")

            Carrier_Comparison_Combined_Model.loc[x,genotype_order_2[genotype]] = odds_ratio
            Carrier_Comparison_Combined_Model.loc[x,genotype_order_3[genotype]] = ("<1e-300" if p_value < 1e-300 
            else f"{p_value:.2e}" if p_value < 0.001 
            else f"{p_value:.3f}") ## Very small values due to high amounts of samples 

            if isinstance(odds_ratio, str):
                if ((odds_ratio == '∞ (Strong Positive Association)') and (float(p_value)< 0.05)):
                    Carrier_Comparison_Combined_Model.loc[x,genotype_order_4[genotype]] = 'yes'
                else: 
                    Carrier_Comparison_Combined_Model.loc[x,genotype_order_4[genotype]] = 'no'
            else: 
                if ((odds_ratio > 1) and (float(p_value) < 0.05)):
                    Carrier_Comparison_Combined_Model.loc[x,genotype_order_4[genotype]] = 'yes'
                else: 
                    Carrier_Comparison_Combined_Model.loc[x,genotype_order_4[genotype]] = 'no'

Carrier_Comparison_Combined_Model

In [ ]:
# Create a combined table comparing results between pwCF and CF Carriers 
combined_table_Combined_Model = pd.DataFrame(columns= ('SNP Locus', 'Alleles', 'Intron', 'Mean Contribution of Main SNP', 'Associated SNP Locus', 'Correlation', 'eQTL? (GTEx v9)', 'SNP ID (GTEx v9)', 'Gene to which the eQTL is associated', 'Tissue where the eQTL was identified', 'GTEx (v9) P-Value of eQTL', 'GTEx (v9) NES of eQTL', 'SNP found in carriers?', 
                                        'SNP dosage associated with V/V & No F508del Mutation', 'pwCF: Odds-Ratio (V/V & No F508del)', 'pwCF: P-Value (V/V & No F508del)', 'pwCF: Significant? (V/V & No F508del)', 'Carriers: Odds-Ratio (V/V & No F508del)', 'Carriers: P-Value (V/V & No F508del)', 'Carriers: Significant? (V/V & No F508del)',
                                        'SNP dosage associated with V/M & No F508del Mutation', 'pwCF: Odds-Ratio (V/M & No F508del)', 'pwCF: P-Value (V/M & No F508del)', 'pwCF: Significant? (V/M & No F508del)', 'Carriers: Odds-Ratio (V/M & No F508del)', 'Carriers: P-Value (V/M & No F508del)', 'Carriers: Significant? (V/M & No F508del)',
                                        'SNP dosage associated with M/M & No F508del Mutation', 'pwCF: Odds-Ratio (M/M & No F508del)', 'pwCF: P-Value (M/M & No F508del)', 'pwCF: Significant? (M/M & No F508del)', 'Carriers: Odds-Ratio (M/M & No F508del)', 'Carriers: P-Value (M/M & No F508del)', 'Carriers: Significant? (M/M & No F508del)',
                                        'SNP dosage associated with M/M & Heterozygous F508del Mutation', 'pwCF: Odds-Ratio (M/M & Heterozygous F508del)', 'pwCF: P-Value (M/M & Heterozygous F508del)', 'pwCF: Significant? (M/M & Heterozygous F508del)', 'Carriers: Odds-Ratio (M/M & Heterozygous F508del)', 'Carriers: P-Value (M/M & Heterozygous F508del)', 'Carriers: Significant? (M/M & Heterozygous F508del)',
                                        'SNP dosage associated with M/M & Homozygous F508del Mutation', 'pwCF: Odds-Ratio (M/M & Homozygous F508del)', 'pwCF: P-Value (M/M & Homozygous F508del)', 'pwCF: Significant? (M/M & Homozygous F508del)', 'Carriers: Odds-Ratio (M/M & Homozygous F508del)', 'Carriers: P-Value (M/M & Homozygous F508del)', 'Carriers: Significant? (M/M & Homozygous F508del)',
                                        'SNP dosage associated with V/M & Heterozygous F508del Mutation', 'pwCF: Odds-Ratio (V/M & Heterozygous F508del)', 'pwCF: P-Value (V/M & Heterozygous F508del)', 'pwCF: Significant? (V/M & Heterozygous F508del)', 'Carriers: Odds-Ratio (V/M & Heterozygous F508del)', 'Carriers: P-Value (V/M & Heterozygous F508del)', 'Carriers: Significant? (V/M & Heterozygous F508del)'))

combined_table_Combined_Model['SNP Locus'] = SNP_table_Combined_Model['SNP Locus']
combined_table_Combined_Model['Alleles'] = SNP_table_Combined_Model['Alleles']
combined_table_Combined_Model['Intron'] = SNP_table_Combined_Model['intron']
combined_table_Combined_Model['Mean Contribution of Main SNP'] = SNP_table_Combined_Model['Mean Contribution of Main SNP']
combined_table_Combined_Model['Associated SNP Locus'] = SNP_table_Combined_Model['Associated SNP Locus']
combined_table_Combined_Model['Correlation'] = SNP_table_Combined_Model['Correlation']
combined_table_Combined_Model['SNP found in carriers?'] = Carrier_Comparison_Combined_Model['SNP dosage associated with V/V & No F508del Mutation'].apply(lambda x: 'yes' if x != 'Not found in Carrier Cohort' else 'no')

combined_table_Combined_Model['SNP dosage associated with V/V & No F508del Mutation'] = SNP_table_Combined_Model['SNP dosage associated with V/V & No F508del Mutation']

combined_table_Combined_Model['pwCF: Odds-Ratio (V/V & No F508del)'] = SNP_table_Combined_Model['Odds-Ratio (V/V & No F508del)']
combined_table_Combined_Model['pwCF: P-Value (V/V & No F508del)'] = SNP_table_Combined_Model['P-Value (V/V & No F508del)']
combined_table_Combined_Model['pwCF: Significant? (V/V & No F508del)'] = SNP_table_Combined_Model['Significant? (V/V & No F508del)']

combined_table_Combined_Model['Carriers: Odds-Ratio (V/V & No F508del)'] = Carrier_Comparison_Combined_Model['Odds-Ratio (V/V & No F508del)']
combined_table_Combined_Model['Carriers: P-Value (V/V & No F508del)'] = Carrier_Comparison_Combined_Model['P-Value (V/V & No F508del)']
combined_table_Combined_Model['Carriers: Significant? (V/V & No F508del)'] = Carrier_Comparison_Combined_Model['Significant? (V/V & No F508del)']

combined_table_Combined_Model['SNP dosage associated with V/M & No F508del Mutation'] = SNP_table_Combined_Model['SNP dosage associated with V/M & No F508del Mutation']

combined_table_Combined_Model['pwCF: Odds-Ratio (V/M & No F508del)'] = SNP_table_Combined_Model['Odds-Ratio (V/M & No F508del)']
combined_table_Combined_Model['pwCF: P-Value (V/M & No F508del)'] = SNP_table_Combined_Model['P-Value (V/M & No F508del)']
combined_table_Combined_Model['pwCF: Significant? (V/M & No F508del)'] = SNP_table_Combined_Model['Significant? (V/M & No F508del)']

combined_table_Combined_Model['Carriers: Odds-Ratio (V/M & No F508del)'] = Carrier_Comparison_Combined_Model['Odds-Ratio (V/M & No F508del)']
combined_table_Combined_Model['Carriers: P-Value (V/M & No F508del)'] = Carrier_Comparison_Combined_Model['P-Value (V/M & No F508del)']
combined_table_Combined_Model['Carriers: Significant? (V/M & No F508del)'] = Carrier_Comparison_Combined_Model['Significant? (V/M & No F508del)']

combined_table_Combined_Model['SNP dosage associated with M/M & No F508del Mutation'] = SNP_table_Combined_Model['SNP dosage associated with M/M & No F508del Mutation']

combined_table_Combined_Model['pwCF: Odds-Ratio (M/M & No F508del)'] = SNP_table_Combined_Model['Odds-Ratio (M/M & No F508del)']
combined_table_Combined_Model['pwCF: P-Value (M/M & No F508del)'] = SNP_table_Combined_Model['P-Value (M/M & No F508del)']
combined_table_Combined_Model['pwCF: Significant? (M/M & No F508del)'] = SNP_table_Combined_Model['Significant? (M/M & No F508del)']

combined_table_Combined_Model['Carriers: Odds-Ratio (M/M & No F508del)'] = Carrier_Comparison_Combined_Model['Odds-Ratio (M/M & No F508del)']
combined_table_Combined_Model['Carriers: P-Value (M/M & No F508del)'] = Carrier_Comparison_Combined_Model['P-Value (M/M & No F508del)']
combined_table_Combined_Model['Carriers: Significant? (M/M & No F508del)'] = Carrier_Comparison_Combined_Model['Significant? (M/M & No F508del)']

combined_table_Combined_Model['SNP dosage associated with M/M & Heterozygous F508del Mutation'] = SNP_table_Combined_Model['SNP dosage associated with M/M & Heterozygous F508del Mutation']

combined_table_Combined_Model['pwCF: Odds-Ratio (M/M & Heterozygous F508del)'] = SNP_table_Combined_Model['Odds-Ratio (M/M & Heterozygous F508del)']
combined_table_Combined_Model['pwCF: P-Value (M/M & Heterozygous F508del)'] = SNP_table_Combined_Model['P-Value (M/M & Heterozygous F508del)']
combined_table_Combined_Model['pwCF: Significant? (M/M & Heterozygous F508del)'] = SNP_table_Combined_Model['Significant? (M/M & Heterozygous F508del)']

combined_table_Combined_Model['Carriers: Odds-Ratio (M/M & Heterozygous F508del)'] = Carrier_Comparison_Combined_Model['Odds-Ratio (M/M & Heterozygous F508del)']
combined_table_Combined_Model['Carriers: P-Value (M/M & Heterozygous F508del)'] = Carrier_Comparison_Combined_Model['P-Value (M/M & Heterozygous F508del)']
combined_table_Combined_Model['Carriers: Significant? (M/M & Heterozygous F508del)'] = Carrier_Comparison_Combined_Model['Significant? (M/M & Heterozygous F508del)']

combined_table_Combined_Model['SNP dosage associated with M/M & Homozygous F508del Mutation'] = SNP_table_Combined_Model['SNP dosage associated with M/M & Homozygous F508del Mutation']

combined_table_Combined_Model['pwCF: Odds-Ratio (M/M & Homozygous F508del)'] = SNP_table_Combined_Model['Odds-Ratio (M/M & Homozygous F508del)']
combined_table_Combined_Model['pwCF: P-Value (M/M & Homozygous F508del)'] = SNP_table_Combined_Model['P-Value (M/M & Homozygous F508del)']
combined_table_Combined_Model['pwCF: Significant? (M/M & Homozygous F508del)'] = SNP_table_Combined_Model['Significant? (M/M & Homozygous F508del)']

combined_table_Combined_Model['Carriers: Odds-Ratio (M/M & Homozygous F508del)'] = Carrier_Comparison_Combined_Model['Odds-Ratio (M/M & Homozygous F508del)']
combined_table_Combined_Model['Carriers: P-Value (M/M & Homozygous F508del)'] = Carrier_Comparison_Combined_Model['P-Value (M/M & Homozygous F508del)']
combined_table_Combined_Model['Carriers: Significant? (M/M & Homozygous F508del)'] = Carrier_Comparison_Combined_Model['Significant? (M/M & Homozygous F508del)']

combined_table_Combined_Model['SNP dosage associated with V/M & Heterozygous F508del Mutation'] = SNP_table_Combined_Model['SNP dosage associated with V/M & Heterozygous F508del Mutation']

combined_table_Combined_Model['pwCF: Odds-Ratio (V/M & Heterozygous F508del)'] = SNP_table_Combined_Model['Odds-Ratio (V/M & Heterozygous F508del)']
combined_table_Combined_Model['pwCF: P-Value (V/M & Heterozygous F508del)'] = SNP_table_Combined_Model['P-Value (V/M & Heterozygous F508del)']
combined_table_Combined_Model['pwCF: Significant? (V/M & Heterozygous F508del)'] = SNP_table_Combined_Model['Significant? (V/M & Heterozygous F508del)']

combined_table_Combined_Model['Carriers: Odds-Ratio (V/M & Heterozygous F508del)'] = Carrier_Comparison_Combined_Model['Odds-Ratio (V/M & Heterozygous F508del)']
combined_table_Combined_Model['Carriers: P-Value (V/M & Heterozygous F508del)'] = Carrier_Comparison_Combined_Model['P-Value (V/M & Heterozygous F508del)']
combined_table_Combined_Model['Carriers: Significant? (V/M & Heterozygous F508del)'] = Carrier_Comparison_Combined_Model['Significant? (V/M & Heterozygous F508del)']

#combined_table_Combined_Model['SNP Locus'] = combined_table_Combined_Model['SNP Locus'].apply(lambda x: x.split(" ")[0])

In [ ]:
# Annotate combined table for eQTLs status 
combined_table_Combined_Model['eQTL? (GTEx v9)'] = combined_table_Combined_Model['SNP Locus'].isin(eQTLs).apply(lambda x: 'yes' if x == True else 'no')

# Coherency check: Making sure eQTL Alleles are correct
print(combined_table_Combined_Model['eQTL? (GTEx v9)'].value_counts())

for x in range(len(combined_table_Combined_Model)):
    if combined_table_Combined_Model.loc[x, 'eQTL? (GTEx v9)'] == 'yes':
        snp_alleles = combined_table_Combined_Model.loc[x, 'Alleles']
        ref_alleles = str(eQTL_catalogue[eQTL_catalogue['Ref'] == combined_table_Combined_Model.loc[x,'SNP Locus']]['Alleles'].iloc[0])
        if snp_alleles != ref_alleles:
            combined_table_Combined_Model.loc[x,'eQTL? (GTEx v9)'] = 'no'
        elif snp_alleles == ref_alleles:
            combined_table_Combined_Model.loc[x,'SNP ID (GTEx v9)'] = str(eQTL_catalogue[eQTL_catalogue['Ref'] == combined_table_Combined_Model.loc[x,'SNP Locus']]['SNP Id'].iloc[0])
            combined_table_Combined_Model.loc[x,'Gene to which the eQTL is associated'] = str(eQTL_catalogue[eQTL_catalogue['Ref'] == combined_table_Combined_Model.loc[x,'SNP Locus']]['Gene Symbol'].iloc[0])
            combined_table_Combined_Model.loc[x,'Tissue where the eQTL was identified'] = str(eQTL_catalogue[eQTL_catalogue['Ref'] == combined_table_Combined_Model.loc[x,'SNP Locus']]['Tissue'].iloc[0])
            combined_table_Combined_Model.loc[x,'GTEx (v9) P-Value of eQTL'] = str(eQTL_catalogue[eQTL_catalogue['Ref'] == combined_table_Combined_Model.loc[x,'SNP Locus']]['P-Value'].iloc[0])
            combined_table_Combined_Model.loc[x,'GTEx (v9) NES of eQTL'] = str(eQTL_catalogue[eQTL_catalogue['Ref'] == combined_table_Combined_Model.loc[x,'SNP Locus']]['NES'].iloc[0])

# Based on the printed values, the eQTL alleles are coherent 
print(combined_table_Combined_Model['eQTL? (GTEx v9)'].value_counts())

In [ ]:
# Check how many SNPs are missing in the Carrier Cohort (Combined Model)
combined_table_Combined_Model['SNP found in carriers?'].value_counts()

In [ ]:
combined_table_Combined_Model[combined_table_Combined_Model['SNP found in carriers?'] == 'no']
# These are indeed not found in the carrier cohort alleles 

In [ ]:
# Check which SNPs were significant between pwCF and Carriers (V/V & No F508del Mutation Class)
combined_table_Combined_Model[['pwCF: Significant? (V/V & No F508del)', 'Carriers: Significant? (V/V & No F508del)']].value_counts()

# Total SNPs validated for association with V/V & No F508del Mutation are 73

In [ ]:
# Check which SNPs were significant between pwCF and Carriers (V/M & No F508del Mutation Class)
combined_table_Combined_Model[['pwCF: Significant? (V/M & No F508del)', 'Carriers: Significant? (V/M & No F508del)']].value_counts()

# Total SNPs validated for association with V/M & No F508del Mutation are 77

In [ ]:
# Check which SNPs were significant between pwCF and Carriers (M/M & No F508del Mutation Class)
combined_table_Combined_Model[['pwCF: Significant? (M/M & No F508del)', 'Carriers: Significant? (M/M & No F508del)']].value_counts()

# Total SNPs validated for association with M/M & No F508del Mutation are 85

In [ ]:
# Check which SNPs were significant between pwCF and Carriers (M/M & Heterozygous F508del Mutation Class)
combined_table_Combined_Model[['pwCF: Significant? (M/M & Heterozygous F508del)', 'Carriers: Significant? (M/M & Heterozygous F508del)']].value_counts()

# Total SNPs validated for association with M/M & Heterozygous F508del Mutation are 139

In [ ]:
# Check which SNPs were significant between pwCF and Carriers (M/M & Homozygous F508del Mutation Class)
combined_table_Combined_Model[['pwCF: Significant? (M/M & Homozygous F508del)', 'Carriers: Significant? (M/M & Homozygous F508del)']].value_counts()

# Total SNPs validated for association with M/M & Homozygous F508del Mutation are 71

In [ ]:
# Amount of SNPs associated with M/M & Homozygous F508del Mutation Class, for which surrogate association was tested in V/V & Heterozygous F508del
len(combined_table_Combined_Model[combined_table_Combined_Model['SNP dosage associated with M/M & Homozygous F508del Mutation'] == 'Homozygous Polymorphism'])

In [ ]:
# Check which SNPs were significant between pwCF and Carriers (V/M & Heterozygous F508del Mutation Class)
combined_table_Combined_Model[['pwCF: Significant? (V/M & Heterozygous F508del)', 'Carriers: Significant? (V/M & Heterozygous F508del)']].value_counts()

# Total SNPs validated for association with V/M & Heterozygous F508del Mutation are 112

In [ ]:
# of Combined Model eQTLs
condition_1 = (combined_table_Combined_Model['pwCF: Significant? (V/V & No F508del)'] == 'yes') & (combined_table_Combined_Model['Carriers: Significant? (V/V & No F508del)'] == 'yes')
condition_2 = (combined_table_Combined_Model['pwCF: Significant? (V/M & No F508del)'] == 'yes') & (combined_table_Combined_Model['Carriers: Significant? (V/M & No F508del)'] == 'yes')
condition_3 = (combined_table_Combined_Model['pwCF: Significant? (M/M & No F508del)'] == 'yes') & (combined_table_Combined_Model['Carriers: Significant? (M/M & No F508del)'] == 'yes')
condition_4 = (combined_table_Combined_Model['pwCF: Significant? (M/M & Heterozygous F508del)'] == 'yes') & (combined_table_Combined_Model['Carriers: Significant? (M/M & Heterozygous F508del)'] == 'yes')
condition_5 = (combined_table_Combined_Model['pwCF: Significant? (M/M & Homozygous F508del)'] == 'yes') & (combined_table_Combined_Model['Carriers: Significant? (M/M & Homozygous F508del)'] == 'yes')
condition_6 = (combined_table_Combined_Model['pwCF: Significant? (V/M & Heterozygous F508del)'] == 'yes') & (combined_table_Combined_Model['Carriers: Significant? (V/M & Heterozygous F508del)'] == 'yes')
Combined_Model_eQTLs = combined_table_Combined_Model[condition_1 | condition_2 | condition_3 | condition_4 | condition_5 | condition_6]
Combined_Model_eQTLs = Combined_Model_eQTLs['SNP Locus']
Combined_Model_eQTLs = Combined_Model_eQTLs[Combined_Model_eQTLs.isin(eQTLs)]
print(len(Combined_Model_eQTLs))

In [ ]:
#combined_table_Combined_Model.to_excel('Combined Associations | Master File.xlsx', index=False)
combined_table_Combined_Model

## Intersection of eQTLs across models

In [ ]:
## Check Intersection of V470M and F508del SNPs
Intersecting_eQTLs_V470M_F508del = set(F508del_eQTLs).intersection(set(V470M_eQTLs))

# There are 48 eQTLs associated with both V470M and F508del
len(Intersecting_eQTLs_V470M_F508del)

In [ ]:
## Check Intersection of V470M and Combined Model SNPs
Intersecting_eQTLs_V470M_Combined= set(Combined_Model_eQTLs).intersection(set(V470M_eQTLs))

# There are 94 eQTLs associated with both V470M and Combined Model
len(Intersecting_eQTLs_V470M_Combined)

In [ ]:
## Check Intersection of F508del and Combined Model SNPs
Intersecting_eQTLs_F508del_Combined= set(Combined_Model_eQTLs).intersection(set(F508del_eQTLs))

# There are 73 eQTLs associated with both F508del and Combined Model
len(Intersecting_eQTLs_F508del_Combined)

In [ ]:
## Check Intersection of SNPs across all models
Intersecting_eQTLs_All = set(F508del_eQTLs).intersection(set(V470M_eQTLs)).intersection(Combined_Model_eQTLs)

# There are 47 eQTLs associated with all of the models
len(Intersecting_eQTLs_All)

## Unique eQTLs found to be associated in each model

In [ ]:
## eQTLs unique to V470M Model
Unique_V470M_eQTLs = set(V470M_eQTLs).difference(Intersecting_eQTLs_V470M_Combined.union(Intersecting_eQTLs_V470M_F508del))
print(len(Unique_V470M_eQTLs))
combined_table_V470M[combined_table_V470M['SNP Locus'].isin(Unique_V470M_eQTLs)]

In [ ]:
## eQTLs unique to F508del Model
Unique_F508del_eQTLs = set(F508del_eQTLs).difference(Intersecting_eQTLs_F508del_Combined.union(Intersecting_eQTLs_V470M_F508del))
print(len(Unique_F508del_eQTLs))
combined_table_f508del[combined_table_f508del['SNP Locus'].isin(Unique_F508del_eQTLs)]

In [ ]:
## eQTLs unique to Combined Model
Unique_Combined_Model_eQTLs = set(Combined_Model_eQTLs).difference(Intersecting_eQTLs_F508del_Combined.union(Intersecting_eQTLs_V470M_Combined))
print(len(Unique_Combined_Model_eQTLs))
combined_table_Combined_Model[combined_table_Combined_Model['SNP Locus'].isin(Unique_Combined_Model_eQTLs)]

## Intron 1 Specific Analysis (Comparing with CFFT Labs)

In [ ]:
## Intron 1 specific overlap (comparing results to CFFT Labs)
F508del_subset = combined_table_f508del[combined_table_f508del['Intron'] == 1]['SNP Locus']
F508del_subset = F508del_subset[F508del_subset.isin(F508del_eQTLs)]

V470M_subset = combined_table_V470M[combined_table_V470M['Intron'] == 1]['SNP Locus']
V470M_subset = V470M_subset[V470M_subset.isin(V470M_eQTLs)]

F508del_V470M_SNPS_intron1 = set(V470M_subset).intersection(set(F508del_subset))
F508del_V470M_SNPS_intron1


In [ ]:
# F508del only eQTLs (Intron 1)
F508del_only_intron1 = set(F508del_subset).difference(set(V470M_subset))
F508del_only_intron1

# No unique eQTLs

In [ ]:
# V470M only eQTLs (Intron 1)
V470M_only_intron1 = set(V470M_subset).difference(set(F508del_subset))
V470M_only_intron1

In [ ]:
# Combined Model only eQTLs in Intron 1 (not found in any of the individual models)
Combined_subset = combined_table_Combined_Model[combined_table_Combined_Model['Intron'] == 1]['SNP Locus']
Combined_subset = Combined_subset[Combined_subset.isin(Combined_Model_eQTLs)]

set(Combined_subset).difference(set(V470M_subset).union(set(F508del_subset)))

# 2 Unique eQTLs